In [ ]:
import random
from typing import Callable, Set, Any

Funkcja hashująca

In [ ]:
class SimpleHash:
    def __init__(self):
        self._p = 2**61 - 1
        self._a= random.randint(1, self._p - 1)
        self._b = random.randint(0, self._p - 1)

    def __call__(self, item: Any) -> float:
        base_hash = abs(hash(str(item)))
        hashed_val = (self._a * base_hash + self._b) % self._p
        return hashed_val / self._p

Zadanie 33

In [7]:
class KMinimumValuesSketch:
    def __init__(self, k: int, hashf: Callable[[Any], float]):
        self._k = k
        self._hash = hashf
        self._min_hashes: Set[float] = set()

    def add(self, item: Any):
        h = self._hash(item)

        if len(self._min_hashes) < self._k:
            self._min_hashes.add(h)
            return

        max_hash = max(self._min_hashes)
        if h < max_hash and h not in self._min_hashes:
            self._min_hashes.remove(max_hash)
            self._min_hashes.add(h)

    def estimate(self) -> float:
        """
        KMV Unbiased Estimator for full buffer: (k - 1) / (k-th minimum value)
        """
        if len(self._min_hashes) < self._k:
            return float(len(self._min_hashes))

        k_th_min = max(self._min_hashes)
        return 0.0 if k_th_min == 0 else (self._k - 1) / k_th_min

In [10]:
k = 400
true_n_unique = 100_000
n_trials = 10

# print(f"k = {k}")
# print(f"True Cardinality = {true_n_unique}")

estimations = list()
errors = list()

for i in range(n_trials):
    kmv = KMinimumValuesSketch(k=k, hashf=SimpleHash())
    stream = list()
    for _ in range(5):
        stream.extend(range(true_n_unique))
    random.shuffle(stream)

    for item in stream:
        kmv.add(item)

    estimated_n_unique = kmv.estimate()
    rel_error = (abs(estimated_n_unique - true_n_unique) / true_n_unique) * 100

    # print(f"--- Run {i} ---")
    # print(f"Estimated Cardinality: {estimated_n_unique:.2f}")
    # print(f"Relative Error: {rel_error:.2f}%")

    estimations.append(estimated_n_unique)
    errors.append(rel_error)

print("--- Summary ---")
print(f"Mean Estimated Cardinality: {(sum(estimations) / n_trials):.2f}")
print(f"Mean Relative Error: {(sum(errors) / n_trials):.2f}%")

--- Summary ---
Mean Estimated Cardinality: 103509.11
Mean Relative Error: 4.13%


Zadanie 34.3

In [ ]:
class SplitKMVSketch:
    def __init__(self, k: int, hashf: Callable[[Any], float]):
        if k % 2 != 0:
            raise ValueError("k must be even")

        self._k = k
        self._sub_k = k // 2
        self._hash = hashf

        self._min_hashes_0: Set[float] = set()
        self._min_hashes_1: Set[float] = set()

    def add(self, item: Any):
        h = self._hash(item)

        direction = abs(hash(str(item))) % 2
        target_hashes = self._min_hashes_0 if direction == 0 else self._min_hashes_1

        if len(target_hashes) < self._sub_k:
            target_hashes.add(h)
            return

        max_hash = max(target_hashes)
        if h < max_hash and h not in target_hashes:
            target_hashes.remove(max_hash)
            target_hashes.add(h)

    def on_get(self) -> float:
        if len(self._min_hashes_0) < self._sub_k or len(self._min_hashes_1) < self._sub_k:
            return 0.0

        l0 = max(self._min_hashes_0) # k-th smalles in first buffer
        l1 = max(self._min_hashes_1) # k-th smalles in second buffer
        return l0 + l1

    def estimate(self) -> float:
        """
        E[L0] ~ k/N and E[L1] ~ k/N => E[Sum] ~ 2k/N => N ~ 2k / Sum.
        """
        val_sum = self.on_get()
        if val_sum == 0.0:
            return float(len(self._min_hashes_0) + len(self._min_hashes_1))

        return (2 * self._k - 2) / val_sum

In [11]:
k = 400
true_n_unique = 100_000
n_trials = 10

# print(f"k = {k} (Split: {k//2} per bucket)")
# print(f"True Cardinality = {true_n_unique}")

estimations = list()
errors = list()

for i in range(n_trials):
    kmv = SplitKMVSketch(k=k, hashf=SimpleHash())

    stream = list()
    for _ in range(5):
        stream.extend(range(true_n_unique))
    random.shuffle(stream)

    for item in stream:
        kmv.add(item)

    estimated_n_unique = kmv.estimate()
    rel_error = (abs(estimated_n_unique - true_n_unique) / true_n_unique) * 100

    # print(f"--- Run {i} ---")
    # print(f"L0 + L1 Sum: {kmv.on_get():.6f}")
    # print(f"Estimated Cardinality: {estimated_n_unique:.2f}")
    # print(f"Relative Error: {rel_error:.2f}%")

    estimations.append(estimated_n_unique)
    errors.append(rel_error)

print("--- Summary ---")
print(f"Mean Estimated Cardinality: {(sum(estimations) / n_trials):.2f}")
print(f"Mean Relative Error: {(sum(errors) / n_trials):.2f}%")

--- Summary ---
Mean Estimated Cardinality: 100347.29
Mean Relative Error: 2.88%
